In [3]:
import pandas as pd
import duckdb
from google.colab import userdata

# Load token
HF_TOKEN = userdata.get('HF_TOKEN')

# Connect
con = duckdb.connect()

# Install and load httpfs
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Set token as a secret (different syntax)
con.execute(f"SET s3_access_key_id = '{HF_TOKEN}';")
con.execute(f"SET s3_secret_access_key = '{HF_TOKEN}';")
con.execute("SET s3_endpoint = 'huggingface.co';")
con.execute("SET s3_use_ssl = true;")

In [7]:
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    print(f" Token found! First 10 chars: {token[:10]}...")
    print(f"Token length: {len(token)}")
except Exception as e:
    print(" Token not found. Please save it in Secrets.")

 Token found! First 10 chars: hf_FjYKaRG...
Token length: 37


In [8]:
import pandas as pd
import duckdb
from google.colab import userdata

# Get token
HF_TOKEN = userdata.get('HF_TOKEN')

# Connect
con = duckdb.connect()

# Install and load httpfs
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Set Hugging Face token using the correct syntax
con.execute(f"""
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print(" Connected to Hugging Face!")

# Test: Count rows
result = con.execute("""
    SELECT COUNT(*) as total_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
""").fetchdf()

print(f" Total rows: {result['total_rows'][0]:,}")

 Connected to Hugging Face!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 Total rows: 78,835,655


In [11]:
# Check column names
result = con.execute("""
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    LIMIT 1
""").fetchdf()

print("Available columns:")
print(result.columns.tolist())

Available columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [13]:
print("Query 1: The Grain (one row = page per day)")
result1 = con.execute("""
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    LIMIT 10
""").fetchdf()
print(result1)

Query 1: The Grain (one row = page per day)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id report_date  gsc_impressions  gsc_clicks  \
0  content_b7e512995f79d5a6  2026-03-01               20           0   
1  content_05597932fe4da067  2026-03-01                1           0   
2  content_7a105f548d9c6916  2026-03-01              125           1   
3  content_905aa32a0230694e  2026-03-01                7           0   
4  content_a3ea9792f793ec72  2026-03-01               11           0   
5  content_36c36abc7650d7af  2026-03-01              239           1   
6  content_a7da352b73b02668  2026-03-01              191           0   
7  content_05434271b257bb68  2026-03-01               55           0   
8  content_d056587ff7faca0c  2026-03-01               77           0   
9  content_bfd1e41c2af250c8  2026-03-01                2           0   

   gsc_avg_position  
0          3.350000  
1          0.000000  
2          4.928000  
3          4.000000  
4          2.272727  
5          7.347280  
6          7.832461  
7          3.272727  
8        

In [14]:
print("\n Query 2: Row Count and Date Span")
result2 = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as first_date,
        MAX(report_date) as last_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").fetchdf()
print(result2)


 Query 2: Row Count and Date Span


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows first_date  last_date
0     9841378 2026-03-01 2026-03-31


In [15]:
print("\nQuery 3: Availability Check")
result3 = con.execute("""
    SELECT
        COUNT(*) as total,
        COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END) as has_impressions,
        COUNT(CASE WHEN gsc_clicks > 0 THEN 1 END) as has_clicks,
        COUNT(CASE WHEN gsc_avg_position IS NOT NULL THEN 1 END) as has_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").fetchdf()
print(result3)


Query 3: Availability Check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     total  has_impressions  has_clicks  has_position
0  9841378          3611061      417981       3611061


## Section 1: The Contract

**1. What one row means for my lane:**
One row = one page (content_hash_id) on one day (report_date).

**2. Which table(s) I'll use:**
fact_content_daily_performance (daily metrics)

**3. Which time window:**
March 2026 (2026-03-01 to 2026-03-31)

**4. What I'll predict or rank:**
Pages with biggest CTR gap (expected CTR based on position vs actual CTR)

**5. One thing I deliberately exclude:**
Pages with less than 100 impressions (too much noise)

In [17]:
## Section 4: The Leakage Trap

# Build feature frame
feature_frame = con.execute("""
    SELECT
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    LIMIT 1000
""").fetchdf()

# Calculate CTR
feature_frame['ctr'] = feature_frame['gsc_clicks'] / feature_frame['gsc_impressions'].replace(0, 1)

# Add a leaky column (like trend_pct in Notebook 02)
feature_frame['leaky_future_ctr'] = feature_frame['ctr'] * 1.5
feature_frame['leaky_opportunity'] = feature_frame['leaky_future_ctr'] - feature_frame['ctr']

print("Leaky column added")
print(feature_frame[['ctr', 'leaky_future_ctr', 'leaky_opportunity']].head())

# Delete the leaky column
feature_frame = feature_frame.drop(['leaky_future_ctr', 'leaky_opportunity'], axis=1)
print("\nLeaky columns removed. Now the data is honest.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leaky column added
     ctr  leaky_future_ctr  leaky_opportunity
0  0.000             0.000              0.000
1  0.000             0.000              0.000
2  0.008             0.012              0.004
3  0.000             0.000              0.000
4  0.000             0.000              0.000

Leaky columns removed. Now the data is honest.


## Section 5: One Limitation

**Limitation:** I'm only using March 2026 data. Results might not generalize to other months. Also, I'm calculating CTR from clicks/impressions, which can be noisy for low-volume pages.

## Section 6: Self-Check

1. Did I define what one row means?  Yes
2. Did I run all three verification queries?  Yes
3. Did I build 5 features with "available when?" lines?  Yes
4. Did I show the leakage trap and remove it?  Yes
5. Did I name one limitation?  Yes